In [1]:
import pandas as pd
combined_df=pd.read_csv("D:/Project/Filtered/Project_Data.csv", low_memory=False)

In [3]:
combined_df['gross_charge'] = pd.to_numeric(combined_df['gross_charge'], errors='coerce')
combined_df['insurer_price'] = pd.to_numeric(combined_df['insurer_price'], errors='coerce')
combined_df['max_price'] = pd.to_numeric(combined_df['max_price'], errors='coerce')
combined_df['discounted_cash'] = pd.to_numeric(combined_df['discounted_cash'], errors='coerce')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# cal best price
def get_best_price(row):
    insurer = row['insurer_price']
    if pd.notna(insurer) and insurer > 0:
        return insurer
    fallback_prices = [row['discounted_cash'], row['gross_charge']]
    fallback_prices = [p for p in fallback_prices if pd.notna(p) and p > 0]
    return min(fallback_prices) if fallback_prices else float('nan')

# get zipcode first
zip_code_input = input("Enter a zip code: ").strip()

# select searching mode
search_mode = input("Search by 'billing_code' or 'description'? ").strip().lower()
while search_mode not in ['billing_code', 'description']:
    search_mode = input("Invalid choice. Please enter 'billing_code' or 'description': ").strip().lower()

# prompt based on search mode
if search_mode == 'billing_code':
    search_value_input = input("Enter a billing code: ").strip()
else:
    desc_keyword = input("Enter a keyword to search in descriptions: ").strip().lower()
    matching_descriptions = combined_df[
        (combined_df['zip_code'] == zip_code_input) &
        (combined_df['description'].str.contains(desc_keyword, case=False, na=False))
    ]['description'].dropna().unique()

    if len(matching_descriptions) == 0:
        print(f"No descriptions containing '{desc_keyword}' found in zip code {zip_code_input}.")
        exit()

    print("\nMatching descriptions:")
    for i, desc in enumerate(matching_descriptions, 1):
        print(f"{i}. {desc}")

    choice = input(f"Select a description by entering the number (1-{len(matching_descriptions)}): ").strip()
    while not choice.isdigit() or not (1 <= int(choice) <= len(matching_descriptions)):
        choice = input(f"Invalid input. Enter a number between 1 and {len(matching_descriptions)}: ").strip()
    
    search_value_input = matching_descriptions[int(choice) - 1]

# enter payer name (optional)
payer_name_input = input("Enter an insurance provider (optional): ").strip()

# filter values based on input given
filtered_df = combined_df[
    (combined_df[search_mode] == search_value_input) &
    (combined_df['zip_code'] == zip_code_input)
].copy()

filtered_df['best_price'] = filtered_df.apply(get_best_price, axis=1)

# data check
if filtered_df.empty:
    print(f"No data found for the given {search_mode} and zip code.")
else:
    # case 1: payer name given → top 3 cheapest plans within that payer
    if payer_name_input:
        payer_df = filtered_df[
            (filtered_df['payer_name'].str.contains(payer_name_input, case=False, na=False)) &
            (filtered_df['insurer_price'].notna()) &
            (filtered_df['insurer_price'] > 0)
        ][['plan_name', 'insurer_price']].dropna().drop_duplicates(subset='insurer_price')

        if payer_df.empty:
            print(f"No valid plans found for '{payer_name_input}' in the given {search_mode} and zip code.")
        else:
            top_3_plans = payer_df.sort_values('insurer_price').head(3)
            print(f"\nTop 3 Cheapest Plans by '{payer_name_input}' for {search_mode.title()} '{search_value_input}' in Zip Code {zip_code_input}:")
            print(top_3_plans)

            plt.figure(figsize=(8, 6))
            sns.barplot(data=top_3_plans, x='plan_name', y='insurer_price', palette='mako')
            plt.title(f"Top 3 Cheapest Plans - {payer_name_input}")
            plt.xlabel("Plan Name")
            plt.ylabel("Insurer Price")
            plt.xticks(rotation=45)
            plt.tight_layout()
            plt.show()

    # case 2: payer name not given → top 3 cheapest payers 
    else:
        payer_summary = filtered_df[
            (filtered_df['insurer_price'].notna()) & 
            (filtered_df['insurer_price'] > 0)
        ].groupby('payer_name')['insurer_price'].mean().reset_index()

        if payer_summary.empty:
            print(f"No insurer prices found in the given zip code and {search_mode}.")
        else:
            top_3_payers = payer_summary.sort_values('insurer_price').head(3)
            print(f"\nTop 3 Cheapest Payers for {search_mode.title()} '{search_value_input}' in Zip Code {zip_code_input}:")
            print(top_3_payers)

            plt.figure(figsize=(8, 6))
            sns.barplot(data=top_3_payers, x='payer_name', y='insurer_price', palette='crest')
            plt.title(f"Top 3 Payers with Lowest Average Insurer Price")
            plt.xlabel("Payer")
            plt.ylabel("Average Insurer Price")
            plt.xticks(rotation=45)
            plt.tight_layout()
            plt.show()
